# Exercise: Prophet Changepoint Analysis

Compare Prophet with default changepoint detection vs manually specified changepoints on electricity demand. Measure out-of-sample MAE.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error


In [ ]:
import matplotlib as mpl

UB = {"Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF"}
UN = {"Black": "#0A0B0F", "Dark Gray": "#5A5C62"}
mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
DATA_PATH = "../data/electricity_demand.csv"
HORIZON = 12


In [ ]:
def load_prophet_df(path):
    """
    Load electricity demand into Prophet format with ds, y, avg_temp_f.
    """
    df = pd.read_csv(path, parse_dates=...)
    df = df.rename(columns=...)
    return df[['ds', 'y', 'avg_temp_f']]

full_df = load_prophet_df(DATA_PATH)
train = full_df.iloc[:-HORIZON]
test = full_df.iloc[-HORIZON:]
print(f"Train: {len(train)} months, Test: {len(test)} months")


In [ ]:
def fit_default_prophet(train):
    """
    Fit Prophet with default changepoint detection and temperature regressor.
    """
    m = Prophet(yearly_seasonality=..., weekly_seasonality=..., daily_seasonality=...)
    m.add_regressor(...)
    m.fit(train)
    return m

default_model = fit_default_prophet(train)
print("Default model fitted")


In [ ]:
def fit_custom_changepoint_prophet(train, changepoints):
    """
    Fit Prophet with manually specified changepoints.
    """
    m = Prophet(changepoints=..., yearly_seasonality=..., weekly_seasonality=..., daily_seasonality=...)
    m.add_regressor(...)
    m.fit(train)
    return m

custom_model = fit_custom_changepoint_prophet(train, changepoints=['2019-01-01', '2021-01-01'])
print("Custom model fitted")


In [ ]:
def forecast_and_compare(default_model, custom_model, train, test):
    """
    Forecast both models and compute test MAE.
    """
    future_default = default_model.make_future_dataframe(periods=..., freq=...)
    future_default["avg_temp_f"] = list(train["avg_temp_f"]) + list(test["avg_temp_f"])
    fc_default = default_model.predict(future_default).iloc[-len(test):]

    future_custom = custom_model.make_future_dataframe(periods=..., freq=...)
    future_custom["avg_temp_f"] = list(train["avg_temp_f"]) + list(test["avg_temp_f"])
    fc_custom = custom_model.predict(future_custom).iloc[-len(test):]

    mae_default = mean_absolute_error(test["y"], fc_default["yhat"])
    mae_custom = mean_absolute_error(test["y"], fc_custom["yhat"])
    return mae_default, mae_custom

mae_def, mae_cust = forecast_and_compare(default_model, custom_model, train, test)
print(f"Default MAE: {mae_def:,.0f}")
print(f"Custom MAE: {mae_cust:,.0f}")


In [ ]:
def plot_comparison(train, test, default_model, custom_model):
    """
    Plot actuals vs both forecasts.
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(train["ds"], train["y"], color=UN["Black"], label="Train")
    ax.plot(test["ds"], test["y"], color=UB["Brand Blue"], label="Actual")
    ax.plot(test["ds"], default_model.predict(default_model.make_future_dataframe(periods=HORIZON, freq="MS")).iloc[-HORIZON:]["yhat"], color=US["Orange"], linestyle="--", label="Default")
    ax.plot(test["ds"], custom_model.predict(custom_model.make_future_dataframe(periods=HORIZON, freq="MS")).iloc[-HORIZON:]["yhat"], color=US["Green"], linestyle="--", label="Custom Changepoints")
    ax.set_title("Prophet: Default vs Custom Changepoints", fontsize=18, fontweight="bold")
    ax.set_ylabel("Demand (MWh)", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_comparison(train, test, default_model, custom_model)
